In [16]:
# -*- coding: utf-8 -*-
"""
Chrome 離線恐龍遊戲 (chrome://dino) 自動高分腳本 (MSS 高速版 + 時序校正)
停止方式：終端機按 Ctrl+C，或將滑鼠快速甩至螢幕左上角 (PyAutoGUI FailSafe)
"""
import subprocess
import sys
import time


def ensure(pkg, mod=None):
    """沒裝就自動安裝，讓 VS Code / Jupyter 按執行就能用"""
    try:
        return __import__(mod or pkg)
    except ImportError:
        print(f"[安裝] 正在安裝 {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        return __import__(mod or pkg)


ensure("pillow", "PIL")
ensure("mss", "mss")
pyg = ensure("pyautogui")

import mss
import mss.tools

pyg.PAUSE = 0        # 取消預設延遲
pyg.FAILSAFE = True  # 滑鼠移到左上角可緊急停止

# ================= 核心參數區 (已針對太晚跳優化) =================
ROW_LOW = 12      # 偵測高度：仙人掌、低飛鳥 → 跳
ROW_MID = 55      # 偵測高度：中飛鳥 → 趴
ROW_HIGH = 78     # 偵測高度：高飛鳥 (站著就能過)
LOOKAHEAD = 450   # 往前看的距離 (從 380 加大到 450，提早發現障礙物)
THRESHOLD = 35    # 與背景灰階差超過多少算障礙物

START_SPEED = 360.0   # 起始速度 (單位/秒)
SPEED_GAIN = 3.8      # 每秒加速量
MAX_SPEED = 950.0     # 最高速度上限

JUMP_LEAD = 0.24      # 起跳提前量(秒) (調大以解決太晚起跳問題)
DUCK_LEAD = 0.22      # 趴下提前量(秒)
JUMP_HOLD = 0.06      # 空白鍵按住時間(秒)
JUMP_COOLDOWN = 0.20  # 起跳後冷卻(秒)
DUCK_HOLD = 0.20      # 鳥離開偵測區後，繼續趴多久(秒)
FREEZE_TIMEOUT = 1.3  # 畫面靜止超過幾秒視為死亡，自動重開
# ==============================================================================


def pick(msg, sec=3):
    print(msg)
    for i in range(sec, 0, -1):
        print(f"   {i} ...", end="\r")
        time.sleep(1)
    p = pyg.position()
    print(f"   已記錄座標：({p.x}, {p.y})      ")
    return p


def main():
    print("=== Chrome 恐龍自動衝高分腳本 (MSS 高速完美版) ===")
    print("請先開好恐龍遊戲，讓恐龍站在地面上。\n")
    head = pick("【1/2】把滑鼠移到『恐龍頭部最前端 (右上角)』")
    foot = pick("【2/2】把滑鼠移到『恐龍腳底的地面』")

    unit = (foot.y - head.y) / 47.0
    if unit < 0.3:
        print("\n[錯誤] 校準失敗：頭的位置要在腳的上方，請重新執行。")
        return
    print(f"\n[成功] 校準完成！1 單位 = {unit:.2f} 像素")

    # 偵測區域計算
    screen_w, screen_h = pyg.size()
    left = int(head.x + unit * 2)
    width = max(10, min(int(LOOKAHEAD * unit), screen_w - left - 1))
    top = max(0, int(foot.y - (ROW_HIGH + 5) * unit))
    bottom = min(screen_h - 1, int(foot.y + 4 * unit))
    height = bottom - top

    def get_row_y(units):
        return int(foot.y - units * unit) - top

    y_low = get_row_y(ROW_LOW)
    y_mid = get_row_y(ROW_MID)
    y_high = get_row_y(ROW_HIGH)

    monitor = {"top": top, "left": left, "width": width, "height": height}

    print("\n3 秒後開始，請不要動滑鼠。Ctrl+C 或滑鼠甩到左上角可停止。")
    time.sleep(3)
    pyg.click(foot.x, foot.y)  # 點擊取得視窗焦點
    pyg.press("space")         # 啟動遊戲

    start_time = time.time()
    next_jump = 0.0
    space_up_at = 0.0
    ducking = False
    duck_until = 0.0
    fast_drop_at = 0.0
    latency = 0.005
    last_sig = None
    same_since = time.time()

    with mss.mss() as sct:
        try:
            while True:
                try:
                    t0 = time.perf_counter()
                    now = time.time()

                    # 放開空白鍵控制
                    if space_up_at and now >= space_up_at:
                        pyg.keyUp("space")
                        space_up_at = 0.0

                    # 自動空中快速下壓落地 (Fast Drop 技巧)
                    if fast_drop_at and now >= fast_drop_at:
                        pyg.keyDown("down")
                        pyg.keyUp("down")
                        fast_drop_at = 0.0

                    # 超高速截圖與格式轉換
                    sct_img = sct.grab(monitor)
                    img_bytes = bytes(sct_img.raw)  # 轉為不可變 bytes 避免 hash 報錯
                    img_w, img_h = sct_img.size

                    # 畫面靜止檢測 (遊戲結束/凍結自動重啟)
                    sig = hash(img_bytes)
                    if sig == last_sig:
                        if now - same_since > FREEZE_TIMEOUT:
                            pyg.press("space")
                            start_time = time.time()
                            same_since = time.time()
                            next_jump = 0.0
                            ducking = False
                            pyg.keyUp("down")
                            pyg.keyUp("space")
                            continue
                    else:
                        last_sig = sig
                        same_since = now

                    step = max(1, int(unit * 1.5))

                    def get_gray(x, y):
                        offset = (y * img_w + x) * 4
                        return (img_bytes[offset] + img_bytes[offset + 1] + img_bytes[offset + 2]) // 3

                    # 背景色動態中位數計算
                    sample_y = min(max(0, y_low), img_h - 1)
                    samples = sorted(get_gray(x, sample_y) for x in range(0, img_w, step * 2))
                    bg = samples[len(samples) // 2]

                    def scan_row_band(center_y):
                        """帶狀高度掃描，避免單點掃描縫隙漏檢"""
                        check_ys = [center_y - 2, center_y, center_y + 2]
                        valid_ys = [y for y in check_ys if 0 <= y < img_h]
                        for x in range(0, img_w, step):
                            for y in valid_ys:
                                if abs(get_gray(x, y) - bg) > THRESHOLD:
                                    return x
                        return None

                    low = scan_row_band(y_low)
                    mid = scan_row_band(y_mid)
                    high = scan_row_band(y_high)

                    # 動態速度估算
                    speed = min(START_SPEED + SPEED_GAIN * (now - start_time), MAX_SPEED) * unit

                    # 1. 地面障礙 -> 跳躍
                    if low is not None and low <= speed * (JUMP_LEAD + latency) and now >= next_jump:
                        if ducking:
                            pyg.keyUp("down")
                            ducking = False

                        pyg.keyDown("space")
                        space_up_at = now + JUMP_HOLD

                        current_cooldown = max(0.18, JUMP_COOLDOWN - (speed / (MAX_SPEED * unit)) * 0.08)
                        next_jump = now + current_cooldown
                        fast_drop_at = now + 0.32  # 壓跳時間稍微拉長，避免落地撞牆

                    # 2. 中飛鳥 -> 趴下
                    elif (mid is not None and high is None and low is None 
                          and mid <= speed * (DUCK_LEAD + latency)):
                        if not ducking:
                            pyg.keyDown("down")
                            ducking = True
                        duck_until = now + DUCK_HOLD

                    # 3. 障礙物離開 -> 解除趴下
                    elif ducking and now >= duck_until:
                        pyg.keyUp("down")
                        ducking = False

                    # 動態計算傳輸與演算延遲 (平滑濾波)
                    loop_time = time.perf_counter() - t0
                    latency = 0.8 * latency + 0.2 * loop_time

                except pyg.FailSafeException:
                    raise
                except Exception as e:
                    print(f"[警告] 執行中發生例外: {e}，準備重試...")
                    time.sleep(0.2)

        except (KeyboardInterrupt, pyg.FailSafeException):
            print("\n[系統] 程式已安全終止。")
        finally:
            pyg.keyUp("space")
            pyg.keyUp("down")


if __name__ == "__main__":
    main()

=== Chrome 恐龍自動衝高分腳本 (MSS 高速完美版) ===
請先開好恐龍遊戲，讓恐龍站在地面上。

【1/2】把滑鼠移到『恐龍頭部最前端 (右上角)』
   已記錄座標：(1085, 508)      
【2/2】把滑鼠移到『恐龍腳底的地面』
   已記錄座標：(1056, 585)      

[成功] 校準完成！1 單位 = 1.64 像素

3 秒後開始，請不要動滑鼠。Ctrl+C 或滑鼠甩到左上角可停止。


C:\Users\mayni\AppData\Local\Temp\ipykernel_17664\4173747693.py:105: DeprecationWarning: mss.mss is deprecated and will be removed in a future release; use mss.MSS instead
  with mss.mss() as sct:



[系統] 程式已安全終止。
